# Load libraries

In [ ]:
import gc
import ctypes

libc = ctypes.CDLL("libc.so.6")

import numpy as np
import pandas as pd

In [ ]:
def free_memory():
    
    _ = gc.collect()
    libc.malloc_trim(0)
#     torch.cuda.empty_cache()

In [ ]:
COMP_LOC = "/kaggle/input/child-mind-institute-detect-sleep-states/"

# Events data

In [ ]:
%%time

PATH = COMP_LOC + "train_events.csv"
events_df = pd.read_csv(PATH)

print(events_df.shape)
events_df.head(2)

In [ ]:
free_memory()

# Remove nulls

In [ ]:
events_df.isnull().sum()

In [ ]:
events_df = events_df[events_df['step'].notnull()]
events_df.drop_duplicates(inplace=True)
events_df = events_df.reset_index(drop=True)
events_df.shape

In [ ]:
events_df.isnull().sum()

# GroupKFold

In [ ]:
from sklearn.model_selection import GroupKFold

# Initialize GroupKFold
gkf = GroupKFold(n_splits=7)

# Create a new 'fold' column and set it to -1 initially
events_df['fold'] = -1

# Split the data into folds
for fold, (train_idx, val_idx) in enumerate(gkf.split(events_df, groups=events_df['series_id'])):
    events_df.loc[val_idx, 'fold'] = fold + 1 


In [ ]:
events_df['fold'].value_counts().sort_index()

In [ ]:
events_df.head()

In [ ]:
events_df[events_df['series_id']=='038441c925bb']['fold'].value_counts()

In [ ]:
events_df.to_csv("train_folds.csv", index=False)